# Working with alternative Sources + Sinks in Apache Spark

## Overview of Sinks in Spark
In Apache Spark, a "sink" refers to the destination for data that is being processed. Sinks can be files, databases, or even real-time dashboards. When working with batch data (DataFrames) and streaming data (Spark Streaming), sinks are essential for storing or outputting the final processed results.

## Sinks in Batch Processing
In batch processing, DataFrames are used to perform operations on static data. After processing, the results often need to be stored persistently. Spark supports various output formats and systems as sinks for DataFrames, including:

- **File systems** (e.g., HDFS, S3)
- **Database systems** (e.g., JDBC databases like PostgreSQL)
- **Big Data systems** (e.g., HBase, Cassandra)

## Integrating JDBC with Apache Spark for PostgreSQL

### Introduction to JDBC in Spark
JDBC (Java Database Connectivity) is an API for the Java programming language that defines how a client may access a database. It provides methods for querying and updating data in a database. Apache Spark supports JDBC to allow data read and write operations directly from/to relational databases like PostgreSQL.

### Configuring Spark with PostgreSQL JDBC
To connect Spark with a PostgreSQL database, you need the PostgreSQL JDBC driver. This driver is essential because it enables Spark to interact with the database. Here's how you set up Spark to use the PostgreSQL JDBC driver:

#### Step 1: Include the PostgreSQL JDBC Driver
You need to download the PostgreSQL JDBC driver and make it accessible to Spark. This involves placing the JDBC jar file in a known location and referencing it in your Spark configuration.

In [ ]:
import os

# Path to the PostgreSQL JDBC jar file
rel_path = os.path.relpath('./libs/postgresql-42.7.3.jar')

We won't need kafka right now but we'll initizalize it for later use

In [ ]:
# Kafka configuration parameters for Confluent Cloud
# Kafka configuration parameters for Confluent Cloud
kafka_params = {
      "kafka.bootstrap.servers": "46.225.20.89:9092",
      "subscribe": "commerce-fhtw",
      "kafka.security.protocol": "PLAINTEXT",
      "startingOffsets": "latest",
}

### Configure your spark session to include custom jar files

In Apache Spark, external libraries, such as JDBC drivers or connectors for other data sources like Apache Kafka, can be included in your Spark jobs by configuring the Spark session

In [ ]:
import os
from pyspark.sql import SparkSession

# Create a Spark session and configure it to use the JDBC jar
spark = SparkSession.builder \
    .appName("PostgreSQL Integration Example") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
    .config("spark.jars", rel_path) \
    .config("spark.executor.extraClassPath", rel_path) \
    .config("spark.driver.extraClassPath", rel_path) \
    .getOrCreate()

### Step 2: Define Connection Properties

To connect to PostgreSQL, specify the connection properties including the URL, user credentials, and the JDBC driver:

In [ ]:
# Connection URL and properties for PostgreSQL
url = "jdbc:postgresql://fhtw-big-data.postgres.database.azure.com/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}

## Enhancing Data Operations with JDBC and Spark DataFrames

### Reading Data from PostgreSQL Using JDBC in Apache Spark
When pulling data from a PostgreSQL database using JDBC with Spark, the process can be fine-tuned to suit specific needs, such as performance optimization and minimizing network overhead. You can perform direct table reads or use custom SQL queries to control the dataset size and structure before it is even loaded into Spark.

### Direct Table Reads
Direct table reads are straightforward; they involve loading the entire table into a DataFrame. This method is simple but might not be efficient if the table is large or if only a subset of the data is needed.

```python
# Loading an entire table
full_table_df = spark.read \
    .jdbc(url=url, table="full_table_name", properties=postgres_options)
full_table_df.show()

In [ ]:
full_table_df = spark.read \
    .jdbc(url=url, table="tracks", properties=postgres_options)
full_table_df.show()

## Custom SQL Queries

Custom SQL queries allow more granular control over the data read operation. By defining a specific SQL query, you can filter and process data within the database before it reaches Spark, which can significantly enhance performance.

In [ ]:
# Reading data using a custom SQL query
query = "(SELECT id, name FROM tracks WHERE composer = 'AC/DC') AS subquery"
filtered_df = spark.read.jdbc(url=url, table=query, properties=postgres_options)
filtered_df.show()

### Managed Tables
When you save a DataFrame as a managed table, Spark manages both the metadata and the data of the table. If the table is deleted, both the metadata and the data are also deleted.

In [ ]:
df = spark.read.json('./data/music_log.json')
df.show(10)

## Using saveAsTable with DataFrames in Spark

The saveAsTable method in Spark is used to write a DataFrame as a table in the Spark SQL catalog. This can either create a new table or overwrite an existing one. It’s a key feature for managing data within Spark’s metastore and supports both managed and external tables.



### Considerations for Using saveAsTable

* Table Formats: You can specify various formats like Parquet, ORC for storing the table. Parquet is often used due to its efficiency in both storage and query performance within Spark.
* Save Modes: The mode option in saveAsTable can be set to “append”, “overwrite”, “ignore”, or “errorifexists”, providing flexibility depending on the operational requirements.


In [ ]:
# Writing DataFrame as a managed table
spark.sql("DROP TABLE IF EXISTS managed_table_name")
df.write.format("parquet").saveAsTable("managed_table_name", mode="overwrite")

### Saving to an External Database (e.g., PostgreSQL)

When you wish to write a DataFrame to an external database like PostgreSQL, you would typically use the JDBC API. This involves specifying the format as "jdbc", providing a URL, and other connection properties:

In [ ]:
# Writing DataFrame as an external table to db streaming_data
# the schema name is streaming_data and the table name is STUDENT_ID_json_logs

student_id = 'ENTER YOUR ID' # Replace with your actual student ID
table_name = f"streaming_data.{student_id}_json_logs"

df.write.jdbc(url=url, table=table_name, mode="overwrite", properties=postgres_options)

### Example, analyze songs played by user gender and store into table

If you’re interested in the differences in song preferences between genders, you might group by both song title and user gender.

In [ ]:
from pyspark.sql.functions import desc

# Group by song and gender, then count occurrences
songs_by_gender_df = df.groupBy("song", "gender").count().orderBy("song", desc("count"))

# Show results
songs_by_gender_df.show()    

In [ ]:
table_name = f"streaming_data.{student_id}_songs_by_gender"

songs_by_gender_df.write.jdbc(url=url, table=table_name, mode="overwrite", properties=postgres_options)

In [ ]:
spark.stop()